In [ ]:

import random
from collections import defaultdict, Counter

import numpy as np
import pretty_midi
from tqdm import tqdm

from src.config import pitch_low, pitch_high, GENERATED_MIDI_DIR


def extract_pitch_sequence(midi_path):
    midi = pretty_midi.PrettyMIDI(midi_path)

    pitches = []
    durations = []

    for inst in midi.instruments:
        if inst.is_drum:
            continue

        notes = sorted(inst.notes, key=lambda x: x.start)

        for note in notes:
            pitches.append(note.pitch)
            durations.append(note.end - note.start)

    return pitches, durations


class RandomNoteGenerator:

    def __init__(self):
        self.duration_choices = [0.125, 0.25, 0.5, 1.0]

    def generate(self, n_notes=200, tempo=120):

        midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
        piano = pretty_midi.Instrument(program=0)

        current_time = 0.0

        for _ in range(n_notes):

            pitch = random.randint(pitch_low, pitch_high)
            duration = random.choice(self.duration_choices)
            velocity = random.randint(50, 100)

            note = pretty_midi.Note(
                velocity=velocity,
                pitch=pitch,
                start=current_time,
                end=current_time + duration
            )

            piano.notes.append(note)

            current_time += random.choice([0.0, 0.125, 0.25])

        midi.instruments.append(piano)
        return midi

    def save(self, output_path, n_notes=200):
        midi = self.generate(n_notes=n_notes)
        midi.write(output_path)
        print(f"[INFO] Saved: {output_path}")


class MarkovChainMusic:

    def __init__(self):
        self.transitions = defaultdict(Counter)
        self.duration_pool = []
        self.pitch_vocab = list(range(pitch_low, pitch_high + 1))

    def train(self, midi_paths):

        print("\nTraining Markov Chain...\n")

        for path in tqdm(midi_paths):

            try:
                pitches, durations = extract_pitch_sequence(path)

                self.duration_pool.extend(durations)

                for i in range(len(pitches) - 1):
                    self.transitions[pitches[i]][pitches[i + 1]] += 1

            except Exception as e:
                print(f"[WARNING] {e}")

        print("\nTraining Complete\n")

    def sample_next(self, current_pitch):

        next_notes = self.transitions[current_pitch]

        if not next_notes:
            return random.choice(self.pitch_vocab)

        pitches = list(next_notes.keys())
        counts = np.array(list(next_notes.values()), dtype=np.float32)
        probs = counts / counts.sum()

        return np.random.choice(pitches, p=probs)

    def generate(self, n_notes=300, tempo=120):

        midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
        piano = pretty_midi.Instrument(program=0)

        current_pitch = random.choice(self.pitch_vocab)
        current_time = 0.0

        for _ in range(n_notes):

            duration = random.choice(self.duration_pool) if self.duration_pool else 0.25

            note = pretty_midi.Note(
                velocity=80,
                pitch=current_pitch,
                start=current_time,
                end=current_time + duration
            )

            piano.notes.append(note)

            current_time += duration
            current_pitch = self.sample_next(current_pitch)

        midi.instruments.append(piano)
        return midi

    def save(self, output_path, n_notes=300):
        midi = self.generate(n_notes=n_notes)
        midi.write(output_path)
        print(f"[INFO] Saved: {output_path}")


if __name__ == "__main__":

    import glob
    from src.config import RAW_MIDI_DIR

    midi_paths = glob.glob(
        str(RAW_MIDI_DIR / "**/*.mid"),
        recursive=True
    )

    # -------------------------
    # Random baseline
    # -------------------------
    random_gen = RandomNoteGenerator()

    for i in range(5):
        path = GENERATED_MIDI_DIR / f"random_{i}.mid"
        random_gen.save(path)

    # -------------------------
    # Markov baseline
    # -------------------------
    markov = MarkovChainMusic()
    markov.train(midi_paths)

    for i in range(5):
        path = GENERATED_MIDI_DIR / f"markov_{i}.mid"
        markov.save(path)